In [1]:
import pandas as pd
import geopandas as gpd
#import matplotlib.ticker as mtick
import numpy as np
# from arcgis.gis import *
# gis = GIS()
import os

# Set working directory
os.chdir(r"E:\xtemp\GitStuff\K-12\TDM-INP-K-12-Enrollment")
print("Now working in:", os.getcwd())

c:\Users\andyli\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\andyli\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Now working in: E:\xtemp\GitStuff\K-12\TDM-INP-K-12-Enrollment


In [2]:
#Layer name for K-12 enrollment data
lyrK12 = r"E:\xtemp\GitStuff\K-12\TDM-INP-K-12-Enrollment\UGRC Data\BY2023\Schools_PreKto12.shp"
lyrMAZ = r"E:\xtemp\ABM\PopulationSim\IntermediateFiles1\data-for-andy\microzones_draft_20260610AndyaddedmazidforUtah.shp"

#Column Names
cnLevel    = 'SchoolLeve'
cnName     = 'SchoolName'
cnG00      = 'K'            #kindergarten as grade 0
cnG01      = 'Grade1'
cnG02      = 'Grade2'
cnG03      = 'Grade3'
cnG04      = 'Grade4'
cnG05      = 'Grade5'
cnG06      = 'Grade6'
cnG07      = 'Grade7'
cnG08      = 'Grade8'
cnG09      = 'Grade9'
cnG10      = 'Grade10'
cnG11      = 'Grade11'
cnG12      = 'Grade12'
cnTot      = 'TotalK12'
cnID       = 'SchoolID'
cnGroup    = 'Group'
cnPrivate  = 'PrivateSch'
cnCharter  = 'CharterSch'
cnOnlineS  = 'OnlineScho'

#enrollment groups - school groups
egPrivateCharter = 'PriCha'
egPublic         = 'Public'
egTotal          = 'Enrol'

#enrollment levels - grade groups 
elElem = 'Elem'
elMidl = 'Midl'
elHigh = 'High'

#school levels - combo of original data and new levels
slElem = 'ELEM'
slMidl = 'MID'
slHigh = 'HIGH'
slKt12 = 'K12'
slKth8 = 'K8'
sl7t12 = '712'

#name of joined layers created later
lyrK12MAZ = 'Schools_PreKto12_withMAZ'
lyrMAZEnrol = 'MAZ_20260624_Enrol'


In [3]:
#grade column names array
cnGrades   = [cnG00,cnG01,cnG02,cnG03,cnG04,cnG05,cnG06,cnG07,cnG08,cnG09,cnG10,cnG11,cnG12]

cnGroupEnrol = [egPublic+'_'+elElem,egPublic+'_'+elMidl,egPublic+'_'+elHigh,egPrivateCharter+'_'+elElem,egPrivateCharter+'_'+elMidl,egPrivateCharter+'_'+elHigh]
#display(cnGroupEnrol)

# list of Enrollment levels by Grade and by School_Level
colnames  =  [cnLevel, cnG00 , cnG01 , cnG02 , cnG03 , cnG04 , cnG05 , cnG06 , cnG07 , cnG08 , cnG09 , cnG10 , cnG11 , cnG12 ]
enroldata = [[slElem , elElem, elElem, elElem, elElem, elElem, elElem, elElem, elElem, elElem, elElem, elElem, elElem, elElem],
             [slMidl , elMidl, elMidl, elMidl, elMidl, elMidl, elMidl, elMidl, elMidl, elMidl, elMidl, elMidl, elMidl, elMidl],
             [slHigh , elHigh, elHigh, elHigh, elHigh, elHigh, elHigh, elHigh, elHigh, elHigh, elHigh, elHigh, elHigh, elHigh],
             [slKt12 , elElem, elElem, elElem, elElem, elElem, elElem, elElem, elMidl, elMidl, elMidl, elHigh, elHigh, elHigh],
             [slKth8 , elElem, elElem, elElem, elElem, elElem, elElem, elElem, elMidl, elMidl, elMidl, elMidl, elMidl, elMidl],
             [sl7t12 , elMidl, elMidl, elMidl, elMidl, elMidl, elMidl, elMidl, elMidl, elMidl, elMidl, elHigh, elHigh, elHigh]
            ]
  
# Create the pandas DataFrame
dfEnrollLevels = pd.DataFrame(enroldata, columns = colnames)
print('\nEnrollment Levels by School Level and Grade:')
display(dfEnrollLevels)


Enrollment Levels by School Level and Grade:


,SchoolLeve,K,Grade1,Grade2,Grade3,Grade4,Grade5,Grade6,Grade7,Grade8,Grade9,Grade10,Grade11,Grade12
0,ELEM,Elem,Elem,Elem,Elem,Elem,Elem,Elem,Elem,Elem,Elem,Elem,Elem,Elem
1,MID,Midl,Midl,Midl,Midl,Midl,Midl,Midl,Midl,Midl,Midl,Midl,Midl,Midl
2,HIGH,High,High,High,High,High,High,High,High,High,High,High,High,High
3,K12,Elem,Elem,Elem,Elem,Elem,Elem,Elem,Midl,Midl,Midl,High,High,High
4,K8,Elem,Elem,Elem,Elem,Elem,Elem,Elem,Midl,Midl,Midl,Midl,Midl,Midl
5,712,Midl,Midl,Midl,Midl,Midl,Midl,Midl,Midl,Midl,Midl,High,High,High


In [4]:
# dfK12 = pd.DataFrame.spatial.from_featureclass(lyrK12)


# Bypass the ArcGIS module completely
dfK12 = gpd.read_file(lyrK12)

#display data totals before filtering
print ("Total Enrollment: " + "{:,}".format(dfK12[cnTot].sum()))
print ("Total Schools: "    + "{:,}".format(dfK12.shape[0])    )

#created filtered dataset that removes unnecessary rows

#make copy to preserve original data
dfK12_fltr = dfK12.copy()

#only school with enrollment
dfK12_fltr = dfK12_fltr[dfK12_fltr[cnTot] > 0]
# Print all column names to see how 'SchoolLevel' was truncated
print(dfK12_fltr.columns.tolist())

#remove NONE and PREK schools, since no data
dfK12_fltr = dfK12_fltr[(dfK12_fltr[cnLevel] != 'PREK') &
                        (dfK12_fltr[cnLevel] != 'NONE')]

print ('Total Filtered Schools After NONE/PREK Removed: ' + "{:,}".format(dfK12_fltr.shape[0])
         + ' (' + str(round(dfK12_fltr.shape[0] / dfK12.shape[0] * 100,1)) + '%)')
print ("Total Filtered Enrollment After NONE/PREK Removed: " + "{:,}".format(dfK12_fltr[cnTot].sum()) + 
         ' (' + str(round(dfK12_fltr[cnTot].sum() / dfK12[cnTot].sum() * 100,1)) + '%)')

dfK12_fltr['OnlineScho'] = dfK12_fltr['OnlineScho'].fillna("")

#remove online schools
dfK12_fltr = dfK12_fltr[dfK12_fltr[cnOnlineS] != 'Y']
print ("Total Filtered Schools After Online Removed: " + "{:,}".format(dfK12_fltr.shape[0]) + 
         ' (' + str(round(dfK12_fltr.shape[0] / dfK12_fltr.shape[0] * 100,1)) + '%)')
print ("Total Filtered Enrollment After Online Removed: " + "{:,}".format(dfK12_fltr[cnTot].sum()) + 
         ' (' + str(round(dfK12_fltr[cnTot].sum() / dfK12[cnTot].sum() * 100,1)) + '%)')

Total Enrollment: 663,526
Total Schools: 1,377
['LEAName', 'SchoolName', 'LEANumber', 'LEAID', 'SchoolID', 'SchoolNumb', 'CharterSch', 'PrivateSch', 'OnlineScho', 'NESS', 'GradeLow', 'GradeHigh', 'SchoolLeve', 'SchoolType', 'Address', 'City', 'State', 'ZipCode', 'FederalLoc', 'PhoneNumbe', 'Website', 'DateOpened', 'DateClosed', 'SchoolYear', 'LEATYPE', 'TotalK12', 'K', 'Grade1', 'Grade2', 'Grade3', 'Grade4', 'Grade5', 'Grade6', 'Grade7', 'Grade8', 'Grade9', 'Grade10', 'Grade11', 'Grade12', 'Female', 'Male', 'AmericanIn', 'Black', 'Asian', 'Hispanic', 'MultipleRa', 'PacificIsl', 'White', 'Economical', 'EnglishLea', 'StudentDis', 'Homeless', 'Preschool', 'geometry']
Total Filtered Schools After NONE/PREK Removed: 1,009 (73.3%)
Total Filtered Enrollment After NONE/PREK Removed: 663,526 (100.0%)
Total Filtered Schools After Online Removed: 993 (100.0%)
Total Filtered Enrollment After Online Removed: 644,169 (97.1%)


In [5]:
#Set private and charter schools to 'PriCha' group
dfK12_fltr.loc[((dfK12_fltr[cnPrivate]=='True') |
                (dfK12_fltr[cnCharter]=='True')), cnGroup] = egPrivateCharter

#set non-private and non-charter schools to 'Public' group
dfK12_fltr.loc[((dfK12_fltr[cnPrivate]!='True') &
                (dfK12_fltr[cnCharter]!='True')), cnGroup] = egPublic

display(dfK12_fltr.groupby([cnGroup],as_index=False)
        .agg(NumSchools=(cnTot,'size'),NumStudents=(cnTot,'sum')))

#display(dfK12_filtered.groupby(['Group'],as_index=False).agg({'TotalK12':[np.size]})

,Group,NumSchools,NumStudents
0,PriCha,128,71485
1,Public,865,572684


In [6]:
#make copy to preserve pre reclassification
dfK12_fltr_rc = dfK12_fltr.copy()

#Explicitely reclassify K through 8, K through 12, and 7 through 12
dfK12_fltr_rc.loc[(((dfK12_fltr_rc[cnLevel]==slElem) & (dfK12_fltr_rc[cnG08]>0)) |
                    (dfK12_fltr_rc[cnLevel]==slMidl) & (dfK12_fltr_rc[cnG04]>0)), cnLevel] = slKth8
dfK12_fltr_rc.loc[ ((dfK12_fltr_rc[cnLevel]==slHigh) & (dfK12_fltr_rc[cnG00]>0)), cnLevel] = slKt12
dfK12_fltr_rc.loc[ ((dfK12_fltr_rc[cnLevel]==slHigh) & (dfK12_fltr_rc[cnG07]>0)), cnLevel] = sl7t12

#manual change for Inovations High School only
dfK12_fltr_rc.loc[(dfK12_fltr_rc[cnName]=='Innovations High School'), cnLevel] = slHigh

#display enrollment by grade to get understanding of the distribution of enrollment for each School_Level
print("\nPre-Reclassify:")
display(dfK12_fltr.groupby([cnLevel,cnGroup], as_index=False).agg(SchoolCount=(cnID,'size'),G00=(cnG00,'sum'),G01=(cnG01,'sum'),G02=(cnG02,'sum'),G03=(cnG03,'sum'),G04=(cnG04,'sum'),G05=(cnG05,'sum'),G06=(cnG06,'sum'),G07=(cnG07,'sum'),G08=(cnG08,'sum'),G09=(cnG09,'sum'),G10=(cnG10,'sum'),G11=(cnG11,'sum'),G12=(cnG12,'sum'),Tot=(cnTot,'sum')))

#display enrollment by grade to get understanding of the distribution of enrollment for each School_Level
print("\nPost-Reclassify:")
display(dfK12_fltr_rc.groupby([cnLevel,cnGroup], as_index=False).agg(SchoolCount=(cnID,'size'),G00=(cnG00,'sum'),G01=(cnG01,'sum'),G02=(cnG02,'sum'),G03=(cnG03,'sum'),G04=(cnG04,'sum'),G05=(cnG05,'sum'),G06=(cnG06,'sum'),G07=(cnG07,'sum'),G08=(cnG08,'sum'),G09=(cnG09,'sum'),G10=(cnG10,'sum'),G11=(cnG11,'sum'),G12=(cnG12,'sum'),Tot=(cnTot,'sum')))


Pre-Reclassify:


,SchoolLeve,Group,SchoolCount,G00,G01,G02,G03,G04,G05,G06,G07,G08,G09,G10,G11,G12,Tot
0,ELEM,PriCha,65,4549,4459,4500,4306,4242,3986,3635,1644,1361,286,49,40,25,33082
1,ELEM,Public,542,35609,37150,39138,41805,42099,41832,27636,97,86,0,3,1,2,265458
2,HIGH,PriCha,30,20,21,26,22,26,25,153,999,1024,1964,2359,2374,2109,11122
3,HIGH,Public,155,294,308,341,380,335,337,370,1186,1325,21622,49656,49614,48413,174181
4,K12,PriCha,14,935,943,934,962,966,986,1042,1184,1192,1016,875,737,664,12436
5,K12,Public,12,53,54,51,62,47,56,52,81,93,138,124,241,762,1814
6,MID,PriCha,19,1558,1546,1588,1531,1541,1685,1718,1428,1426,824,0,0,0,14845
7,MID,Public,156,14,11,13,19,15,642,16075,43265,44478,26698,3,0,0,131231



Post-Reclassify:


,SchoolLeve,Group,SchoolCount,G00,G01,G02,G03,G04,G05,G06,G07,G08,G09,G10,G11,G12,Tot
0,712,PriCha,14,0,0,0,0,0,0,126,970,970,1001,928,878,729,5602
1,712,Public,31,0,0,0,0,0,0,15,808,953,1603,2959,3044,2879,12261
2,ELEM,PriCha,35,2535,2528,2448,2418,2282,2147,1804,139,0,0,0,0,0,16301
3,ELEM,Public,537,35484,37042,39001,41696,42003,41702,27520,2,0,0,0,0,0,264450
4,HIGH,PriCha,15,0,0,0,0,0,0,0,0,41,963,1431,1496,1380,5311
5,HIGH,Public,123,0,0,0,1,0,1,2,0,15,19701,46419,46268,45231,157638
6,K12,PriCha,15,955,964,960,984,992,1011,1069,1213,1205,1016,875,737,664,12645
7,K12,Public,13,347,362,392,441,382,392,405,459,450,456,402,543,1065,6096
8,K8,PriCha,48,3572,3477,3640,3419,3501,3420,3445,2829,2684,1110,49,40,25,31211
9,K8,Public,6,139,119,149,128,111,141,116,95,86,0,3,1,2,1090


In [7]:
#investigate
#dfK12_fltr_rc[dfK12_fltr_rc[cnLevel]==sl7t12]

#blue peak high
#dfK12_fltr[dfK12_fltr[cnName]=='Blue Peak High']

#dfK12_fltr_rc[(dfK12_fltr_rc[cnLevel]==slElem) & (dfK12_fltr_rc[cnG12]>0)]

In [7]:
#calculate enrollment by level for all schools

#normalize (reverse-pivot) enrollment by grade for ease of calcs
dfK12_melt = pd.melt(dfK12_fltr_rc, id_vars=[cnID,cnLevel,cnGroup], value_vars=cnGrades)
dfK12_melt = dfK12_melt.rename(columns={'variable':'Grade','value':'Enrollment'})
#display(dfK12_melt)

#create 'lookup table' for Enrollment Level
dfEnrollLevels_melt = pd.melt(dfEnrollLevels, id_vars=[cnLevel], value_vars=cnGrades)
dfEnrollLevels_melt = dfEnrollLevels_melt.rename(columns={'variable':'Grade','value':'EnrollmentLevel'})
#display(dfEnrollLevels_melt)

#join to get enrollment levels
dfK12_enrolllevels = pd.DataFrame.merge(dfK12_melt,dfEnrollLevels_melt,on=(cnLevel,'Grade'))

#group to get subtotals by School, Group, and Enrollment Level
dfK12_enrolllevels = dfK12_enrolllevels.groupby([cnID,cnGroup,'EnrollmentLevel'],as_index=False).agg(Enrollment=('Enrollment','sum'))
dfK12_enrolllevels

,SchoolID,Group,EnrollmentLevel,Enrollment
0,132,Public,Elem,535
1,133,Public,Elem,416
2,134,Public,Elem,531
3,135,Public,Elem,509
4,136,Public,Elem,876
...,...,...,...,...
1142,186822,PriCha,Midl,42
1143,186823,Public,Elem,544
1144,186836,Public,Elem,144
1145,186836,Public,Midl,17


In [8]:

dfK12_enrol = dfK12_enrolllevels.pivot(index=[cnID], columns=[cnGroup,'EnrollmentLevel'],values='Enrollment')

#collapse multi-index and include '_' between index values
dfK12_enrol.columns = ['_'.join(col).strip() for col in dfK12_enrol.columns.values]

dfK12_enrol = dfK12_enrol.fillna(0)
dfK12_enrol = dfK12_enrol.astype(int)

dfK12_enrol = dfK12_enrol[cnGroupEnrol]
display(dfK12_enrol)

display(dfK12_enrol.sum())
display(dfK12_enrol.sum().sum())

,Public_Elem,Public_Midl,Public_High,PriCha_Elem,PriCha_Midl,PriCha_High
SchoolID,,,,,,
132,535,0,0,0,0,0
133,416,0,0,0,0,0
134,531,0,0,0,0,0
135,509,0,0,0,0,0
136,876,0,0,0,0,0
...,...,...,...,...,...,...
186816,0,0,0,521,259,0
186822,0,0,0,167,42,0
186823,544,0,0,0,0,0


Public_Elem    268074
Public_Midl    136082
Public_High    168530
PriCha_Elem     47710
PriCha_Midl     13653
PriCha_High     10122
dtype: int64

644171

In [9]:
#export school enrollment results to csv

# =========================================================
# 1. EXPORT TO CSV (Keeping this since you requested the output)
# =========================================================
strK12File = r"results\k12_enrol06242026.csv"
dfK12_enrol=dfK12_enrol.reset_index()
dfK12_enrol.to_csv(strK12File, index=False)

# =========================================================
# 2. ATTRIBUTE JOIN (Equivalent to AddJoin 'KEEP_ALL')
# =========================================================
# Assuming 'gdf_k12' is your loaded school shapefile/feature class.
# 'how="left"' acts exactly like 'KEEP_ALL', keeping all original school points.
gdf_k12_joined = dfK12.merge(dfK12_enrol, on=cnID, how='left')

# =========================================================
# 3. SPATIAL JOIN (Equivalent to SpatialJoin 'KEEP_COMMON' and 'WITHIN')
# =========================================================
# Assuming 'gdf_taz' is your loaded TAZ shapefile.
# Best practice: Always ensure Coordinate Reference Systems (CRS) match before a spatial join!
gdf_maz = gpd.read_file(lyrMAZ)
if gdf_k12_joined.crs != gdf_maz.crs:
    print(f"Aligning CRS to match TAZs ({gdf_maz.crs})...")
    gdf_k12_joined = gdf_k12_joined.to_crs(gdf_maz.crs)

# how='inner' is the exact equivalent of join_type='KEEP_COMMON'
# predicate='within' is the exact equivalent of match_option='WITHIN'
gdf_k12_maz = gpd.sjoin(gdf_k12_joined, gdf_maz, how='inner', predicate='within')

# =========================================================
# 4. REMOVE JOIN 
# =========================================================
# NOT NEEDED! 
# In GeoPandas, your original `gdf_k12` variable remains completely untouched. 
# You can continue using `gdf_k12` for other operations without needing to "remove" anything!

Aligning CRS to match TAZs (EPSG:26912)...


In [10]:
#get joined school/TAZ layer in order to aggregate to CO_TAZID

sdfK12MAZ = gdf_k12_maz.copy()
dfK12MAZ = sdfK12MAZ[['SchoolName','maz_id'] + cnGroupEnrol]
dfK12MAZ = dfK12MAZ.fillna(0)
#display(dfK12TAZ.columns)
display(dfK12MAZ)

,SchoolName,maz_id,Public_Elem,Public_Midl,Public_High,PriCha_Elem,PriCha_Midl,PriCha_High
0,River Rock School,30162,932.0,0.0,0.0,0.0,0.0,0.0
1,Skyridge High School,30751,0.0,114.0,2500.0,0.0,0.0,0.0
2,Springside School,28877,821.0,0.0,0.0,0.0,0.0,0.0
3,Summit High,32370,0.0,19.0,56.0,0.0,0.0,0.0
4,Alta Independent,31964,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
1368,Summit Academy High School,20651,0.0,0.0,0.0,0.0,0.0,599.0
1369,KoolMinds East Sandy,21006,0.0,0.0,0.0,0.0,0.0,0.0
1370,Square Peg Academy,31793,0.0,0.0,0.0,0.0,0.0,0.0
1373,Mountain Point Academy,30821,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
#aggregate by maz_id
dfMAZSummary = dfK12MAZ.groupby(['maz_id']).agg(sum)
dfMAZSummary[dfMAZSummary.select_dtypes(include='number').columns] = \
    dfMAZSummary.select_dtypes(include='number').astype(int)

#remove rows with all zeros
dfMAZSummary = dfMAZSummary.loc[~(dfMAZSummary==0).all(axis=1)]

dfMAZSummary

C:\Users\andyli\AppData\Local\Temp\ipykernel_53840\2567594038.py:2: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  dfMAZSummary = dfK12MAZ.groupby(['maz_id']).agg(sum)


,SchoolName,Public_Elem,Public_Midl,Public_High,PriCha_Elem,PriCha_Midl,PriCha_High
maz_id,,,,,,,
44,Goldminer's Daughter,11,0,0,0,0,0
6818,Summit Academy - Bluffdale,0,0,0,537,0,0
6858,Blackridge School,782,0,0,0,0,0
6859,Ridge View Elementary,713,0,0,0,0,0
6862,Oak Hollow School,488,0,0,0,0,0
...,...,...,...,...,...,...,...
32152,Eagle Valley School,776,0,0,0,0,0
32156,Just 4 Kids Adventures Orem,0,0,0,0,0,0
32163,Cedar Valley High,0,0,3325,0,0,0


In [12]:
#calculate total enrollment for each level by adding Private/Charter and Public schools
dfMAZSummary[egTotal+'_'+elElem]  = dfMAZSummary[egPublic+'_'+elElem] + dfMAZSummary[egPrivateCharter+'_'+elElem]
dfMAZSummary[egTotal+'_'+elMidl]  = dfMAZSummary[egPublic+'_'+elMidl] + dfMAZSummary[egPrivateCharter+'_'+elMidl]
dfMAZSummary[egTotal+'_'+elHigh]  = dfMAZSummary[egPublic+'_'+elHigh] + dfMAZSummary[egPrivateCharter+'_'+elHigh]
dfMAZSummary

,SchoolName,Public_Elem,Public_Midl,Public_High,PriCha_Elem,PriCha_Midl,PriCha_High,Enrol_Elem,Enrol_Midl,Enrol_High
maz_id,,,,,,,,,,
44,Goldminer's Daughter,11,0,0,0,0,0,11,0,0
6818,Summit Academy - Bluffdale,0,0,0,537,0,0,537,0,0
6858,Blackridge School,782,0,0,0,0,0,782,0,0
6859,Ridge View Elementary,713,0,0,0,0,0,713,0,0
6862,Oak Hollow School,488,0,0,0,0,0,488,0,0
...,...,...,...,...,...,...,...,...,...,...
32152,Eagle Valley School,776,0,0,0,0,0,776,0,0
32156,Just 4 Kids Adventures Orem,0,0,0,0,0,0,0,0,0
32163,Cedar Valley High,0,0,3325,0,0,0,0,0,3325


In [13]:
#check the final data

#subtotals by enrollment group and level
display(dfMAZSummary.select_dtypes(include='number').sum())

#subtotals by enrollment group
print (egPublic        ,':',dfMAZSummary[[egPublic        +'_'+elElem,egPublic        +'_'+elMidl,egPublic        +'_'+elHigh]].sum().sum())
print (egPrivateCharter,':',dfMAZSummary[[egPrivateCharter+'_'+elElem,egPrivateCharter+'_'+elMidl,egPrivateCharter+'_'+elHigh]].sum().sum())
print (egTotal         ,':',dfMAZSummary[[egTotal         +'_'+elElem,egTotal         +'_'+elMidl,egTotal         +'_'+elHigh]].sum().sum())

Public_Elem    199296
Public_Midl     99592
Public_High    122298
PriCha_Elem     39591
PriCha_Midl     11681
PriCha_High      8449
Enrol_Elem     238887
Enrol_Midl     111273
Enrol_High     130747
dtype: int64

Public : 421186
PriCha : 59721
Enrol : 480907


In [15]:
#export CO_TAZID enrollment to csv

sFilename = r'results\K12_Enrollment_public_MAZ.csv'

dfMAZSummary.to_csv(sFilename)
print('CSV Exported to: ' + sFilename)

CSV Exported to: results\K12_Enrollment_public_MAZ.csv


In [16]:
import geopandas as pd
import pandas as pd

# Assuming 'lyrMAZ' is a string variable with your MAZ name (used for your CSV filename)
# and 'gdf_maz' is your already-loaded MAZ GeoDataFrame.

# =========================================================
# 1. LOAD THE CSV
# =========================================================
csv_path = f"results\K12_Enrollment_public_MAZ.csv"
df_enrollment = pd.read_csv(csv_path)

# =========================================================
# 2. ATTRIBUTE JOIN (Equivalent to AddJoin 'KEEP_ALL')
# =========================================================
# how='left' keeps all MAZ polygons, even if they don't have enrollment data in the CSV
gdf_maz_joined = gdf_maz.merge(df_enrollment, on='maz_id', how='left')

# =========================================================
# 3. EXPORT TO SHAPEFILE (Equivalent to CopyFeatures)
# =========================================================
output_shapefile = r"results\maz_with_enrollment.shp"
gdf_maz_joined.to_file(output_shapefile)

# =========================================================
# 4. REMOVE JOIN
# =========================================================
# NOT NEEDED! 
# Your original `gdf_maz` variable remains completely untouched.

C:\Users\andyli\AppData\Local\Temp\ipykernel_53840\1131592502.py:23: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_maz_joined.to_file(output_shapefile)
